건물 위 기지국(Tx)과 도로 위를 달리는 자동차(Rx) 시나리오

반복문(Loop)을 통해 Rx(자동차)의 위치(Position)와 방향(Orientation)을 프레임마다 갱신하고, compute_paths()를 호출하여 채널을 얻는 것

In [ ]:
import tensorflow as tf
import numpy as np
import sionna
from sionna.rt import load_scene, Transmitter, Receiver, PlanarArray, PathSolver, SceneObject, ITURadioMaterial
from sionna.rt import Camera
import time

In [ ]:
# 1. 3D 도심지 환경 로드 (Sionna 내장 'etoile' 맵 사용 - 건물 환경)
# 실제 연구시에는 load_scene("my_city_model.xml")로 변경
scene = load_scene(sionna.rt.scene.etoile)

안테나 설정

In [ ]:
# 주파수 대역 설정
scene.frequency = 15e9
# 1. 기지국용 (Tx): 8x8 배열, 이중 편파
# "dual" -> "VH" (수직/수평) 또는 "cross" (+45/-45) 사용
tx_array = PlanarArray(
    num_rows=8, 
    num_cols=8, 
    polarization="VH",  # <--- "dual" 대신 "VH" 또는 "cross"
    pattern="iso"
)

# 2. 차량용 (Rx): 단일 안테나, 단일 편파
# "single" -> "V" (수직) 또는 "H" (수평) 사용
rx_array = PlanarArray(
    num_rows=1, 
    num_cols=1, 
    polarization="V",   # <--- "single" 대신 "V"
    pattern="iso"
)

# Scene에 적용
scene.tx_array = tx_array
scene.rx_array = rx_array

In [ ]:
# 2. Solver 인스턴스 생성 (이 녀석이 Ray Tracing을 수행합니다)
solver = PathSolver()

Tx(기지국) 배치: 건물 위라고 가정하여 높이(z)를 30m로 설정

In [ ]:
tx_pos = [0, -50, 30] 
tx = Transmitter(name="BaseStation", position=tx_pos)
scene.add(tx)

Rx(자동차) 궤적 생성 (Trajectory)

In [ ]:
# 도로를 따라 달리는 경로 생성 (예: x축 -100m 에서 +100m로 이동)
num_time_steps = 20  # 시뮬레이션 할 스텝 수
x_coords = np.linspace(-100, 100, num_time_steps)
y_coords = np.zeros(num_time_steps) # y는 0으로 고정 (직선 도로)
z_coords = np.ones(num_time_steps) * 1.5 # 자동차 지붕 높이 (1.5m)

# [Step, 3] 형태의 좌표 배열
car_trajectory = np.stack([x_coords, y_coords, z_coords], axis=1)

# Rx 객체 초기 생성 (첫 번째 위치에 배치)
rx = Receiver(name="Car_Rx", position=car_trajectory[0])
scene.add(rx)

In [ ]:
# 1. 자동차 재질 생성
car_material = ITURadioMaterial("car_metal", "metal", thickness=0.01, color=(0.8, 0.1, 0.1))

# 2. 자동차 객체 생성
car_mesh = SceneObject(fname=sionna.rt.scene.low_poly_car,
                       name="Visual_Car",
                       radio_material=car_material)

# 3. 장면에 추가 (scene.edit 사용)
scene.edit(add=[car_mesh])

# 4. 위치 설정 (.tolist() 필수!)
car_mesh.position = car_trajectory[0].tolist()
car_mesh.scale = [1.5, 1.5, 1.5]

In [ ]:
# 카메라 설정 및 추가
my_cam = Camera(position=[0, -100, 500], look_at=[0, 0, 0])

# 가장 확실한 방법:
scene.render(camera=my_cam, resolution=(640, 480))

시뮬레이션 루프 (Mobility.ipynb 참조)

In [ ]:
print("Simulating Car Movement...")

# 결과를 저장할 리스트
cir_results = [] 

for i, pos in enumerate(car_trajectory):
    rx.position = pos
    car_mesh.position = pos.tolist()
    scene.render(camera=my_cam, resolution=(640, 480))
    
    # 애니메이션 효과를 위해 잠시 대기
    time.sleep(0.1) 
    paths = solver(scene, max_depth=5, samples_per_src=int(1e6))
    
    # C. 채널 임펄스 응답(CIR) 계산
    a, tau = paths.cir()
    
    # [수정] .numpy() 속성이 없을 수 있으므로, 명시적으로 변환
    # a와 tau가 TensorFlow Tensor라면 .numpy()가 먹히지만, 
    # 만약 Dr.Jit Array라면 np.array()로 감싸야 함
    
    a_np = np.array(a)   # 안전하게 numpy 배열로 변환
    tau_np = np.array(tau)
    
    cir_results.append((a_np, tau_np))
    
    # shape 확인도 numpy 배열로 변환 후 하는 것이 안전
    print(f"Step {i+1}/{num_time_steps}: Found {a_np.shape} paths (Shape check)")

print("Simulation Completed.")

결과 분석

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# 1. 데이터 가져오기
a_0 = cir_results[0][0] 
tau_0 = cir_results[0][1] 

# 2. Delay 추출
# tau_0 Shape이 (2, 1, 1, 16) 등으로 예상되므로, 첫 번째 수신기([0])의 경로만 가져옵니다.
# 결과가 (16,) 형태가 되도록 인덱싱
delays = tau_0[0, 0, :]  # 상황에 따라 tau_0[0, 0, 0, :] 일 수도 있으니 shape 확인 필요
# 만약 위의 delays가 (1, 16)이라면 flatten() 필요
delays = delays.flatten()

# 3. Power 계산 수정
a_squeezed = np.squeeze(a_0) 
print(f"DEBUG: a_squeezed shape: {a_squeezed.shape}")

# (2, 128, 16) 형태일 경우를 대비한 로직
if a_squeezed.ndim == 3:
    # 예: [Num_Rx, Num_Tx_Ant, Num_Paths] -> (2, 128, 16)
    # 1) 첫 번째 수신기만 선택 (인덱스 0) -> (128, 16)
    a_selected = a_squeezed[0, :, :]
    
    # 2) 안테나 차원(128) 합치기 (axis=0) -> (16,)
    power = np.sum(np.abs(a_selected)**2, axis=0)
    
elif a_squeezed.ndim == 2:
    # 기존 로직 유지
    num_paths = delays.shape[0]
    if a_squeezed.shape[1] == num_paths:
        power = np.sum(np.abs(a_squeezed)**2, axis=0)
    else:
        power = np.sum(np.abs(a_squeezed)**2, axis=1)
else:
    power = np.abs(a_squeezed)**2

# shape 다시 확인
print(f"DEBUG: delays shape: {delays.shape}")
print(f"DEBUG: power shape: {power.shape}")

# 4. 그래프 그리기
plt.figure(figsize=(10, 6))
plt.stem(delays * 1e9, power) 
plt.xlabel("Delay (ns)")
plt.ylabel("Power (Linear)")
plt.title(f"Power Delay Profile (Num Paths: {len(delays)})")
plt.grid(True, alpha=0.3)
plt.show()